# Chat with a language model

This notebook talks to an Ollama server running in an Apptainer container on your GPU. Start the
server first, in a JupyterLab terminal, as described in
[Section 7.4](../docs/07-containers.md) of the tutorial. The notebook only sends it HTTP requests.

Use the **Detection (workshop)** kernel.

## Find the server

The server listens on a port worked out from your user ID, so that two people on the same node do
not collide. This cell uses the same formula as the terminal commands and checks that something is
answering there.

In [ ]:
import json
import os

import requests

PORT = 11000 + os.getuid() % 1000
URL = f"http://127.0.0.1:{PORT}"

models = requests.get(f"{URL}/api/tags", timeout=5).json()["models"]
for m in models:
    print(f"{m['name']:20s} {m['size'] / 1e9:.1f} GB")

If that fails with `Connection refused`, the server is not running in this session. Go back to the
terminal and start it.

## Ask one question

`ask` streams the reply as it is generated, a few words at a time, the same way chat websites do.
It also returns the timing Ollama reports, so you can see how fast the GPU is producing text.

In [ ]:
MODEL = "llama3.2:3b"


def ask(messages, model=MODEL):
    reply = []
    with requests.post(f"{URL}/api/chat", json={"model": model, "messages": messages}, stream=True) as r:
        r.raise_for_status()
        for line in r.iter_lines():
            chunk = json.loads(line)
            piece = chunk.get("message", {}).get("content", "")
            print(piece, end="", flush=True)
            reply.append(piece)
            if chunk.get("done"):
                rate = chunk["eval_count"] / (chunk["eval_duration"] / 1e9)
                print(f"\n\n[{chunk['eval_count']} tokens at {rate:.0f} tokens/s]")
    return "".join(reply)


ask([{"role": "user", "content": "Explain what a GPU cluster is to a first-year student, in three sentences."}])

## Have a conversation

The model has no memory of its own. Every request has to include the whole conversation so far,
which is what the `history` list is for. Type `quit` to stop.

In [ ]:
history = [{"role": "system", "content": "You are a helpful assistant. Keep answers short."}]

while True:
    question = input("you> ")
    if question.strip().lower() in {"quit", "exit", ""}:
        break
    history.append({"role": "user", "content": question})
    print("model> ", end="")
    history.append({"role": "assistant", "content": ask(history)})
    print()

## Try a bigger model

`qwen2.5:7b` is more than twice the size of `llama3.2:3b`. It is slower and takes more of the
GPU's memory, and its answers are usually better. Run the same question through both and compare.
The first request to a model takes longer, because Ollama has to load it onto the GPU.

In [ ]:
question = [{"role": "user", "content": "What is the difference between a login node and a compute node?"}]
for model in ["llama3.2:3b", "qwen2.5:7b"]:
    print(f"=== {model}")
    ask(question, model=model)

Check how much GPU memory the models are using now:

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)